<a href="https://colab.research.google.com/github/Vali-git/poc_simulation_data_analysis/blob/main/poc_simulation_presale_data_sample.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import re
import numpy as np
from rapidfuzz import fuzz

df = pd.read_csv('https://raw.githubusercontent.com/Vali-git/poc_simulation_data_analysis/refs/heads/main/presales_data_sample.csv')
df.head()

,input_row_key,input_company_name,input_main_country_code,input_main_country,input_main_region,input_main_city,input_main_postcode,input_main_street,input_main_street_number,veridion_id,...,twitter_url,instagram_url,linkedin_url,ios_app_url,android_app_url,youtube_url,tiktok_url,technologies,created_at,last_updated_at
0,0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK,Pakistan,Sindh,Karachi,NaN,NaN,NaN,26e22210-93e5-11eb-b997-8dd98d09cf25,...,NaN,NaN,http://www.linkedin.com/company/mnet-services-...,NaN,NaN,NaN,NaN,web servers: apache http server - 2 | javascri...,2020-02-25T14:47:51.000Z,2024-11-29T04:18:00.109Z
1,0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK,Pakistan,Sindh,Karachi,NaN,NaN,NaN,01004641-1dd8-11ef-9268-316fc8e174dd,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-29T16:24:11.019Z,2025-04-20T15:03:24.026Z
2,0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK,Pakistan,Sindh,Karachi,NaN,NaN,NaN,8266efc1-13e7-11ec-aa14-7bf90e1e10f1,...,https://twitter.com/Network24seven%20,NaN,https://www.linkedin.com/company/51633612%20%20,NaN,NaN,NaN,NaN,NaN,2021-09-12T05:28:48.000Z,2025-03-18T23:08:37.059Z
3,0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK,Pakistan,Sindh,Karachi,NaN,NaN,NaN,0183f0b2-93e5-11eb-be5a-4f810ab55f2e,...,https://twitter.com/emeriosoft,NaN,https://www.linkedin.com/company/emeriosoft,NaN,NaN,NaN,NaN,miscellaneous: popper | maps: google maps | pr...,2020-05-03T12:33:22.000Z,2025-03-31T16:16:58.462Z
4,0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK,Pakistan,Sindh,Karachi,NaN,NaN,NaN,87bb7cde-93e4-11eb-8474-3bbe2d07207d,...,https://twitter.com/AsiaticPR,https://www.instagram.com/asiaticpublicrelations/,https://www.linkedin.com/company/asiaticpublic...,NaN,NaN,https://www.youtube.com/channel/UClKgHvHIuOu4K...,NaN,miscellaneous: babel | javascript libraries: c...,2020-02-19T03:58:25.000Z,2024-11-25T11:35:47.963Z


In [3]:
# Extragem doar coloanele de input și eliminăm duplicatele
# Presupunem că 'input_company_name' este numele căutat
df_unique_inputs = df[['input_company_name', 'input_main_country_code']].drop_duplicates().reset_index(drop=True)

print(f"Ai în total {len(df_unique_inputs)} căutări unice.")
df_unique_inputs

Ai în total 591 căutări unice.


,input_company_name,input_main_country_code
0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,PK
1,2OPERATE A/S,DK
2,ACCENTURE SERVICES AS,NO
3,COMBA TELECOM LIMITED,HK
4,Comengineering Sdn. Bhd.,MY
...,...,...
586,CISCO INTERNATIONAL LIMITED,GB
587,CISCO NORWAY AS,NO
588,CITERO AS,NO
589,"CLOUDERA, INC.",US


In [4]:
import re
from rapidfuzz import fuzz

def calculeaza_scor_prioritate_brand(row):
    #Curățăm inputul și identificăm cuvintele
    text_in = re.sub(r'[^a-z0-9\s]', ' ', str(row['input_company_name']).lower())
    words_in = [w for w in text_in.split() if len(w) > 1]
    if not words_in: return pd.Series([0.0, 0.0, False])

    #Primul cuvânt este Brandul
    brand_word = words_in[0]
    #Restul sunt cuvinte secundare
    secondary_words = words_in[1:]

    #Pregătim candidatul pentru scor
    cand_name = str(row['company_name']).lower()
    text_cand = re.sub(r'[^a-z0-9\s]', ' ', cand_name)
    words_cand = text_cand.split()
    cand_compact = "".join(words_cand)

    #VERIFICARE BRAND (Pondere 80% din scorul de nume)
    #Verificăm dacă primul cuvânt există sau e aproape identic
    brand_match = False
    if brand_word in cand_compact:
        brand_match = True
    else:
        for wc in words_cand:
            if fuzz.ratio(brand_word, wc) >= 90:
                brand_match = True
                break

    if not brand_match:
        return pd.Series([0.0, 0.0, False])

    #Scor bază pentru brand (80% din cele 70 de puncte = 56 puncte)
    puncte_nume = 56.0

    #VERIFICARE CUVINTE SECUNDARE (Pondere 20% din scorul de nume = 14 puncte)
    if secondary_words:
        sec_gasite = 0
        for sw in secondary_words:
            if sw in cand_compact or any(fuzz.ratio(sw, wc) >= 90 for wc in words_cand):
                sec_gasite += 1

        #Adăugăm proporțional din cele 14 puncte rămase
        puncte_nume += (sec_gasite / len(secondary_words)) * 14.0
    else:
        #Dacă nu există cuvinte secundare în input, dăm tot scorul de 70 pe brand
        puncte_nume = 70.0

    #Țara (30 puncte)
    c1 = str(row['input_main_country_code']).strip().lower()
    c2 = str(row['main_country_code']).strip().lower()
    tara_match = (c1 == c2) and (c1 not in ['nan', ''])
    puncte_tara = 30.0 if tara_match else 0.0

    final_score = puncte_nume + puncte_tara
    match_pct = (puncte_nume / 70.0) * 100

    return pd.Series([round(final_score, 2), round(match_pct, 1), tara_match])

df[['final_score', 'nume_match_pct', 'country_match']] = df.apply(calculeaza_scor_prioritate_brand, axis=1)

In [5]:
df[['input_company_name', 'company_name', 'final_score', 'nume_match_pct', 'country_match']].head(50)

,input_company_name,company_name,final_score,nume_match_pct,country_match
0,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,New Millennium Network,0.0,0.0,False
1,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,Private Helipad Ali Villa,0.0,0.0,False
2,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,24seven Research Network,61.6,88.0,False
3,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,EmerioSoft,0.0,0.0,False
4,24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED,Asiatic Public Relations Network,0.0,0.0,False
5,2OPERATE A/S,2operate,100.0,100.0,True
6,2OPERATE A/S,CO 2 Operate,70.0,100.0,False
7,2OPERATE A/S,Conzentrate,0.0,0.0,False
8,2OPERATE A/S,2A Pharma ApS.,0.0,0.0,False
9,2OPERATE A/S,RenSams Solutions,0.0,0.0,False


In [7]:
import pandas as pd

#Filtrare: Doar rândurile care au țară
df_filtrat = df[df['main_country_code'].notna() & (df['main_country_code'].str.strip() != '')].copy()

#Sortare: Cele mai bune match-uri să fie primele
df_sorted = df_filtrat.sort_values(
    by=['input_row_key', 'main_country_code', 'nume_match_pct', 'final_score'],
    ascending=[True, True, False, False]
)


df_final = df_sorted.groupby(['input_row_key', 'main_country_code']).head(5).copy()

#Validare Brand: Minim 50% la nume
df_final = df_final[df_final['nume_match_pct'] >= 50].copy()

#Adăugăm o coloană de Status
def set_status(pct):
    if pct >= 90: return 'EXACT/HIGH'
    if pct >= 60: return 'MEDIUM'
    return 'LOW/PARTIAL'

df_final['match_level'] = df_final['nume_match_pct'].apply(set_status)

#AFIȘARE
print("\n--- PREVIEW: TOP 5 PER ȚARĂ (DOAR COLOANE CHEIE) ---")
coloane_vizibile = [
    'input_company_name',
    'company_name',
    'main_country_code',
    'match_level',
    'nume_match_pct',
    'final_score'
]
print(df_final[coloane_vizibile].head(20).to_string(index=False))

#Se salvează tot tabelul (df_final conține toate coloanele originale)
df_final.to_csv('rezultate_veridion_complet.csv', index=False)




--- PREVIEW: TOP 5 PER ȚARĂ (DOAR COLOANE CHEIE) ---
                      input_company_name             company_name main_country_code match_level  nume_match_pct  final_score
24-SEVEN MEDIA NETWORK (PRIVATE) LIMITED 24seven Research Network                IN      MEDIUM            88.0        61.60
                            2OPERATE A/S                 2operate                DK  EXACT/HIGH           100.0       100.00
                           3 STEP IT A/S           3 Step IT A/S.                DK  EXACT/HIGH           100.0       100.00
                           3 STEP IT A/S             3Step IT Oy.                FI  EXACT/HIGH           100.0        70.00
                           3 STEP IT A/S                  3stepIT                SE  EXACT/HIGH           100.0        70.00
                            3 STEP IT AS           3 Step IT A/S.                DK  EXACT/HIGH           100.0        70.00
                            3 STEP IT AS             3Step IT Oy.      